In [1]:
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

data = pd.read_excel(r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage1\dataset1_729×87.xlsx")
X = data.iloc[:, 3:]
X
feature_names = X.columns
X = StandardScaler().fit_transform(X)

min_val = np.min(X, axis=0)
max_val = np.max(X, axis=0)
X = (X - min_val) / (max_val - min_val)
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(X)
X.shape


(729, 87)

In [2]:
def extract_numbers(file_path):
    df = pd.read_excel(file_path)
    valid_numbers = []
    for index, value in df['composition'].items():
        if not pd.isna(value):
            valid_numbers.append(int(df.at[index, 'number']))
    return valid_numbers

file_path = r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage1\benchmark1.xlsx"
reported_structures = extract_numbers(file_path)
print(reported_structures)

def remove_noise_by_nearest_cluster(X, labels):
    """Assign noise points to the cluster of the nearest non-noise points."""

    noise_mask = labels == -1

    if not any(noise_mask):
        return labels

    core_samples = ~noise_mask
    X_core = X[core_samples]
    y_core = labels[core_samples]

    if len(X_core) == 0:

        new_labels = np.zeros_like(labels)
        return new_labels

    nn = NearestNeighbors(n_neighbors=1).fit(X_core)
    distances, indices = nn.kneighbors(X[noise_mask])

    new_labels = labels.copy()
    new_labels[noise_mask] = y_core[indices.flatten()]
    return new_labels

for j in range(30, 100, 5):
    j = j/100

    pca = PCA(n_components= j)
    pca_data = pca.fit_transform(X)
    num_pca = pca_data.shape[1]

    for eps in range(1,100,1):
        eps = eps/100
        for min_samples in range(6, 21,3):
            db = DBSCAN(eps=eps, metric='cosine', min_samples=min_samples)
            db.fit(pca_data)
            labels = db.labels_
            labels = remove_noise_by_nearest_cluster(pca_data, labels)

            if len(np.unique(labels)) > 1:
                    print(f"\nSilhouette score: {silhouette_score(pca_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(pca_data, labels):.2f}")
            else:
                    pass

            data['cluster_label'] = labels
            unique_labels = np.unique(labels)
            print( f"\nPCA={j}",f"\neps={eps}", f"\nmin_samples={min_samples}", f"\nNumber of clusters={len(unique_labels)}")

            dict = {}

            label_1 = []
            for num  in  reported_structures:
                    label = labels[num-1]
                    label_1.append(int(label))
            print(label_1)

            for label in unique_labels:

                cluster_samples = data[data['cluster_label'] == label]
                total_in_cluster = len(cluster_samples)

                reported_in_cluster = len([num for num in cluster_samples['number']
                                        if num in reported_structures])

                print(f"Cluster {label}: {reported_in_cluster}/{total_in_cluster} "
                    f"(reported ratio: {reported_in_cluster/total_in_cluster:.2%})")


[22, 23, 24, 25, 260, 300, 384, 387, 400, 404, 423, 435, 489, 641, 703]

Silhouette score: 0.44 
Calinski-Harabasz score: 469.59

PCA=0.3 
eps=0.01 
min_samples=6 
Number of clusters=2
[1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1]
Cluster 0: 6/278 (reported ratio: 2.16%)
Cluster 1: 9/451 (reported ratio: 2.00%)

Silhouette score: 0.28 
Calinski-Harabasz score: 267.73

PCA=0.3 
eps=0.01 
min_samples=9 
Number of clusters=3
[1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1]
Cluster 0: 6/251 (reported ratio: 2.39%)
Cluster 1: 9/453 (reported ratio: 1.99%)
Cluster 2: 0/25 (reported ratio: 0.00%)

Silhouette score: 0.28 
Calinski-Harabasz score: 267.73

PCA=0.3 
eps=0.01 
min_samples=12 
Number of clusters=3
[1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1]
Cluster 0: 6/251 (reported ratio: 2.39%)
Cluster 1: 9/453 (reported ratio: 1.99%)
Cluster 2: 0/25 (reported ratio: 0.00%)

Silhouette score: 0.29 
Calinski-Harabasz score: 269.15

PCA=0.3 
eps=0.01 
min_samples=15 
Number of clusters=3
[1, 1, 1, 

In [5]:
pca = PCA(n_components= 0.5)
pca_data = pca.fit_transform(X)
print(pca_data.shape)
ratios = pca.explained_variance_ratio_
components = pca.components_
X = pd.DataFrame(X)

components_df = pd.DataFrame(
    components,
    columns=X.columns,
    index=[f"PC{i+1}" for i in range(components.shape[0])]
)

output_file = "pca_components.xlsx"
components_df.to_excel(output_file, index=True)

db = DBSCAN(eps=0.02, metric='cosine', min_samples=12)
db.fit(pca_data)
labels = db.labels_
labels = remove_noise_by_nearest_cluster(pca_data, labels)

df = pd.DataFrame(labels)
df.to_excel("DBSCAN_result_1.xlsx", index=False)

print(f"\nSilhouette score: {silhouette_score(pca_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(pca_data, labels):.2f}")

data['cluster_label'] = labels
unique_labels = np.unique(labels)

dict = {}

label_1 = []
for num  in  reported_structures:
    label = labels[num-1]
    label_1.append(int(label))
print(label_1)

for label in unique_labels:

    cluster_samples = data[data['cluster_label'] == label]
    total_in_cluster = len(cluster_samples)

    reported_in_cluster = len([num for num in cluster_samples['number']
                                        if num in reported_structures])

    print(f"Cluster {label}: {reported_in_cluster}/{total_in_cluster} "
                    f"(reported ratio: {reported_in_cluster/total_in_cluster:.2%})")

print("Label distribution:", Counter(labels))


(729, 4)

Silhouette score: 0.28 
Calinski-Harabasz score: 212.53
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Cluster 0: 0/168 (reported ratio: 0.00%)
Cluster 1: 15/561 (reported ratio: 2.67%)
Label distribution: Counter({np.int64(1): 561, np.int64(0): 168})


In [6]:
import pandas as pd
from sklearn.decomposition import PCA
from openpyxl.styles import Font
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import Workbook

wb = Workbook()
ws = wb.active

for r in dataframe_to_rows(components_df, index=True, header=True):
    ws.append(r)

red_bold_font = Font(color="FF0000", bold=True)

for row in range(2, ws.max_row + 1):

    values = [ws.cell(row=row, column=col).value for col in range(2, ws.max_column + 1)]

    values = [0 if v is None else v for v in values]

    abs_values = [abs(v) for v in values]
    top_10_indices = sorted(range(len(abs_values)), key=lambda i: abs_values[i], reverse=True)[:10]

    for idx in top_10_indices:
        ws.cell(row=row, column=idx + 2).font = red_bold_font

output_file = "pca_components_highlighted.xlsx"
wb.save(output_file)

print(f"The component-loading matrix has been saved to {output_file}, with the 10 largest absolute values highlighted.")


The component-loading matrix has been saved to pca_components_highlighted.xlsx, with the 10 largest absolute values highlighted.
